Вместо прямой инициализации класса `MappingLoadersFromCSV` и итерирования по объекту этого класса функция `get_mapping_from_dir` позволяет получить привычный словарь.

In [ ]:
def get_mapping_from_dir(csv_object:CSVFilesObject):
    loader = MappingLoadersFromCSV(csv_object)
    
    all_mappings = {}

    for label, mapping in loader:
        all_mappings[label] = mapping

    return all_mappings

Визуализация промежутков возрастания и убывания при условии их чередования, начиная с возрастания.

In [ ]:
for label, mapping in smooth_mappings.items():
    fig, ax = plt.subplots(figsize=(12, 9))
    extreme_points = get_idx_extreme_points(mapping, 5)
    number = 0

    for i_color, point in enumerate(extreme_points):
        if (i_color % 2) == 0:
            color = 'red'
        else:
            color = 'green'

        ax.plot(mapping.get_x()[number:point], mapping.get_y()[number:point], color=color)
        number = point

    ax.grid(visible=True)
    ax.set_title(label)

Разбиение с шагом, равным шагу производной и окном в два отрезка производной.

In [ ]:
def create_section_for_approximation(mapping:Mapping, extreme_idx:list, derivative_size_window:int):
    COUNT_DERIVATIVE_IN_LOCAL_PART = 2

    x = mapping.get_x()
    left_border = 0
    right_border = len(x) - len(x) % derivative_size_window
    extreme_idx.append(right_border)
    print(extreme_idx)

    all_sections = []
    print('left_border', left_border)
    for idx in range(1, len(extreme_idx)):
        local_parts = []
        print(extreme_idx[idx])
        for part in range(left_border, extreme_idx[idx] + 1, derivative_size_window):
            local_parts.append([x[part], x[part + COUNT_DERIVATIVE_IN_LOCAL_PART * derivative_size_window]])

            print('part', part)
        left_border = extreme_idx[idx - 1]
        print('--------------------------')
        print('left_border', left_border)

        all_sections.extend(local_parts)

    return all_sections

Создание дерева папок для отчёта.

In [ ]:
def create_dir_tree(path, main_dir, child_directory:list):
    os.makedirs(f'{path}/{main_dir}', exist_ok=True)

    for child_dir_name in child_directory:
        os.makedirs(f'{path}/{main_dir}/{child_dir_name}')

OUTPUTS_PATH = 'D:/Data-sience/#diplom/approximation/DopplerCurve/outputs'

main_dirs = [label for label in smooth_mappings.keys()]
child_directory = ['lognormal_distribution_main_parts', 'normal_distribution_main_parts', 'inverse_exponential_other_part']
other_dirs = ['images', 'tables']

for name in main_dirs:
    create_dir_tree(OUTPUTS_PATH, name, child_directory)

    for d in child_directory:
        create_dir_tree(f'{OUTPUTS_PATH}/{name}', d, other_dirs)

Цикл по всем временным ряда, разбиение с учётом точек экстремумов, подсчёт метрик и параметров, визуализация каждого промежутка, выгрузка всех результатов в папки.

In [ ]:
visualize = VisualizationPlots((12, 9))

funcs_metrics = {
    'MAE': mean_absolute_error,
    'MAPE': mean_absolute_percentage_error,
    'R^2': r2_score,
    'C_norm': max_error
}

for name, mapping in smooth_mappings.items():
    clear_mapping = preprocessing_pipeline(mapping)

    normalize_obj = MinMaxNormalizeForMappings(clear_mapping)
    normalize_mapping = normalize_obj.normalize()

    informative_slice_mapping, other_mapping = get_informative_slice(normalize_mapping, 0.7)
    
    extreme_points = get_idx_extreme_points(informative_slice_mapping, 5)

    # для сравнения и начальных параметров посчитаем sigma и mu принудительно
    sigma_parametres = calculate_sigma_for_lognormal_distrib(informative_slice_mapping, extreme_points)
    mu_parametres = calculate_time_peak_for_lognormal_distrib(informative_slice_mapping, extreme_points)
    print(len(sigma_parametres))
    print(len(mu_parametres))

    sections_loader = create_parts_from_sections_with_extreme(informative_slice_mapping, extreme_points)

    all_parts = pd.DataFrame()
    all_parametres_info = pd.DataFrame()
    all_errors = pd.DataFrame()

    for i, (parts, sigma, mu) in enumerate(zip(sections_loader, sigma_parametres, mu_parametres)):
        parts_name = pd.DataFrame()
        FREE_TERM = 0.3

        if i % 2 == 0:
            COEF_PEAK = 1
            BOUNDS_FOR_PARAMETRES = ([-1, 0.001, -5, -2], [3, 2, 5, 2]) # Lognormal distrib for positive coef peak
            # BOUNDS_FOR_PARAMETRES = ([-3, 0.001, -5, -2], [3, 5, 3, 2]) # Normal distrib for positive coef peak

        else:
            COEF_PEAK = -1
            BOUNDS_FOR_PARAMETRES = ([-1, 0.001, -5, -2], [0, 2, 5, 2])
            # BOUNDS_FOR_PARAMETRES = ([-3, 0.001, -5, -2], [0, 5, 3, 2]) # Normal distrib for negative coef peak

        names_parts = [f'[{round(part[0], 3)} , {round(part[1], 3)}]' for part in parts]
        parts_name['parts'] = names_parts
        all_parts = pd.concat([all_parts, parts_name], ignore_index=True)
        print(all_parts)
        functions = _get_functions_for_parts(lognormal_distribution, parts)
        parametres_for_parts = [(COEF_PEAK, sigma, mu, FREE_TERM) for _ in range(len(parts))]
        bounds_parametres_for_parts = [BOUNDS_FOR_PARAMETRES for _ in range(len(parts))]

        structure_pipeline = StructurePipelineApproximation(
            names_parts,
            functions,
            parts,
            parametres_for_parts,
            bounds_parametres_for_parts
        )

        pipeline = PipelineLoader(informative_slice_mapping, structure_pipeline)
        results = run_pipeline(pipeline)

        results_denormalize = [denormalize_result_approx_struct(res, normalize_obj) for res in results]
    
        for res, label in zip(results_denormalize, names_parts):
            visualize.create_plot_result_approx(res, [label, 'result']).savefig(rf'D:/Data-sience/#diplom/approximation/DopplerCurve/outputs/{name}/lognormal_distribution_main_parts/images/{i}_{label}.png')
            all_parametres_info = pd.concat([all_parametres_info, res.parametres_show()], ignore_index=True)
            all_errors = pd.concat([all_errors, res.metrix_values_show(funcs_metrics)], ignore_index=True)

    all_parts = pd.concat([all_parts, all_parametres_info, all_errors], axis=1)
    all_parts.to_excel(rf'D:/Data-sience/#diplom/approximation/DopplerCurve/outputs/{name}/lognormal_distribution_main_parts/tables/parametres_metrix.xlsx')

Тактика разбиения отрезков на отрезки аппроксимации

In [ ]:
def _get_directions_for_mapping(mapping:Mapping, size_window:int):
    y = mapping.get_y()
    directions = []

    for idx in range(0, len(y), size_window):
        if len(y) - idx < size_window:
            break

        derivative = np.diff(y[idx:idx + size_window])
        directions.append(np.mean(derivative))

    return directions

def create_parts_from_sections_with_extreme(
        mapping:Mapping, 
        extreme_idx:list, 
        rotate_extreme_idx:int=1, 
        splitting_coefficient:int=3
):  
    last_idx = mapping.shape[0]

    nodes_slice = _get_nodes_slice_for_extreme_idx(extreme_idx, rotate_extreme_idx, splitting_coefficient, last_idx)

    node_border = 0
    for idx in extreme_idx:
        section_border = idx
        sections = []

        for node in range(node_border, len(nodes_slice)):
            if nodes_slice[node] >= section_border:
                break
            part = [mapping.get_x_values_from_idx(nodes_slice[node]), mapping.get_x_values_from_idx(nodes_slice[node + splitting_coefficient])]

            sections.append(part)

        node_border = node

        yield sections

def _get_nodes_slice_for_extreme_idx(
        extreme_idx:list, 
        rotate_extreme_idx:int, 
        splitting_coefficient:int,
        last_idx:int
):
    START_IDX = 0
    nodes_slice_idx = [START_IDX]

    rotate_idxs = _calculate_right_rotate_idxs(extreme_idx, rotate_extreme_idx)
    rotate_idxs.append(last_idx)


    for idx in range(1, len(rotate_idxs)):
        count_points = rotate_idxs[idx] - rotate_idxs[idx - 1]
        window = count_points // splitting_coefficient

        for i, node in enumerate(range(rotate_idxs[idx - 1] + window, rotate_idxs[idx], window)):
            if i == splitting_coefficient - 1:
                break

            nodes_slice_idx.append(node)

        nodes_slice_idx.append(rotate_idxs[idx])

    return nodes_slice_idx

def _calculate_right_rotate_idxs(extreme_idx:list, rotate_extreme_idx:int):

Аппроксимация разными функциями для данных о воде.

In [ ]:
def coeff_search(_data, _right, _left):
    approximate = pd.DataFrame()
    funcs = [lognormal_distribution, normal_distribution, lognormal_distribution_with_normal_distrib, extrapolate_model]
    funcs_cfg = [
        _get_start_parametres_values_and_bounds_for_logn_distrib,
        _get_start_parametres_values_and_bounds_for_normal_distrib,
        _get_start_parametres_values_and_bounds_for_lognormal_with_normal_distrib,
        _get_start_parametres_values_and_bounds_for_extrapolate_model
    ]

    approximate['long'] = df.iloc[1:-1, 1]

    for i in range(0, _data.shape[1]):
        all_info_for_series = pd.DataFrame()

        for func, func_cfg in zip(funcs, funcs_cfg):
            errors = pd.DataFrame()
            parametres = pd.DataFrame()
            all_info_for_series_for_func = pd.DataFrame()

            now_mapping = Mapping(
                df.iloc[1:_data.iloc[:-1, i].count()+1, 1].to_numpy(),
                _data.iloc[0:_data.iloc[:-1, i].count(), i].to_numpy(),
                True
            )

            normalize_obj = MinMaxNormalizeForMappings(now_mapping)

            pipeline_cfg = get_config_for_sliding_approximation(now_mapping, func, func_cfg, 200, 12)

            pipeline = PipelineLoader(now_mapping, pipeline_cfg)

            try:
                all_result = run_pipeline(pipeline)
            except:
                continue

            result_denormalize = [denormalize_result_approx_struct(res, normalize_obj) for res in all_result]

            for res in result_denormalize:
                parametres = pd.concat([parametres, res.parametres_show()])
                errors = pd.concat([errors, res.metrix_values_show(funcs_metrics)])

                all_info_for_series_for_func = pd.concat([parametres, errors], axis=1)

            all_info_for_series_for_func['window_size'] = 200
            all_info_for_series_for_func['step'] = 12
            all_info_for_series_for_func['function'] = func.__name__
            all_info_for_series = pd.concat([all_info_for_series, all_info_for_series_for_func], axis=1)

        all_info_for_series.to_excel(f'waste_water_result_for_{i}.xlsx')

coeff_search(DataAll, [-1 for _ in range(DataAll.shape[1])], np.zeros(DataAll.shape[1]).astype(int))